# Publish Data Product to SAP BDC

Publish all Iceberg V3 tables in `TECHED_DA261.<SCHEMANAME>` back to SAP Business Data Cloud via the zero-copy connector.

**Workflow:**
1. Create share `<SHARENAME>` and grant access to all 12 tables
2. Attach the share to the SAP BDC connector (idempotent — wrapped in `BEGIN/EXCEPTION`)
3. Generate minimal CSN Interop v1.0 with correct Snowflake → Iceberg → CDS type mappings
4. Publish with `SYSTEM$SAP_PUBLISH_DATA_PRODUCT`

> **Note:** Columns with unsupported types (VARIANT, TIMESTAMP_NTZ(9), TIMESTAMP_LTZ(9)) are excluded from the CSN definitions. SAP materialization cannot process these Iceberg types.

## Instructions
- Replace `<SCHEMANAME>` to your Schema Name
- Replace `<SHARENAME>` to a SHARENAME with your user name appended
- Replace `<USERNAME>` with your username in the Data Product Title

In [ ]:
%%sql -r set_role
USE ROLE ACCOUNTADMIN

In [ ]:
%%sql -r set_wh
USE WAREHOUSE DEMOWAREHOUSE

In [ ]:
%%sql -r create_share
BEGIN
    CREATE SHARE IF NOT EXISTS <SHARENAME>;
    GRANT USAGE ON DATABASE TECHED_DA261 TO SHARE <SHARENAME>;
    GRANT USAGE ON SCHEMA TECHED_DA261.<SCHEMANAME> TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.FORECAST_VS_MILESTONES TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.LEARNING_FORECAST_RAW TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.TRANSFORMATION_MILESTONES TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.VW_CDP_FEATURES TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.VW_LABELED_EMPLOYEES TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.VW_LEARNING_FEATURES TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.VW_LEARNING_MONTHLY_SERIES TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.VW_PERFORMANCE_FEATURES TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.VW_SKILLS_SIGNAL TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.VW_SUCCESSION_FEATURES TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.WORKFORCE_READINESS_SCORES TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.WORKFORCE_TRANSFORMATION_VELOCITY TO SHARE <SHARENAME>;
END;

In [ ]:
%%sql -r attach_share
BEGIN
    ALTER ZEROCOPY CONNECTOR DB_BDC_SANDBOX.PUBLIC.SAP_BDC_CONNECT_SANDBOX_BFSNOWFLAKE
        SET SHARE_BACK = TRUE;
    ALTER ZEROCOPY CONNECTOR DB_BDC_SANDBOX.PUBLIC.SAP_BDC_CONNECT_SANDBOX_BFSNOWFLAKE
        ADD SHARE <SHARENAME>;
EXCEPTION
    WHEN OTHER THEN
        RETURN 'Share may already be associated with the connector: ' || SQLERRM;
END;

In [ ]:
import json

csn = {
    "csnInteropEffective": "1.0",
    "$version": "2.0",
    "i18n": {},
    "meta": {
        "creator": "Snowflake CSN Interop Generator - Minimal",
        "flavor": "inferred"
    },
    "definitions": {
        "<SCHEMANAME>": {"kind": "context"},

        "<SCHEMANAME>.FORECAST_VS_MILESTONES": {
            "kind": "entity",
            "elements": {
                "MILESTONE_NAME": {"type": "cds.String", "key": True, "notNull": True},
                "PROJECTED_COMPLETIONS": {"type": "cds.Double"},
                "LOWER_90": {"type": "cds.Double"},
                "UPPER_90": {"type": "cds.Double"},
                "REQUIRED_COMPLETIONS": {"type": "cds.Decimal", "precision": 2, "scale": 0},
                "MILESTONE_STATUS": {"type": "cds.String"}
            }
        },

        "<SCHEMANAME>.LEARNING_FORECAST_RAW": {
            "kind": "entity",
            "elements": {
                "FORECAST": {"type": "cds.Double", "key": True, "notNull": True},
                "LOWER_BOUND": {"type": "cds.Double"},
                "UPPER_BOUND": {"type": "cds.Double"}
            }
        },

        "<SCHEMANAME>.TRANSFORMATION_MILESTONES": {
            "kind": "entity",
            "elements": {
                "DEPARTMENT": {"type": "cds.Decimal", "precision": 20, "scale": 0, "key": True, "notNull": True},
                "MILESTONE_NAME": {"type": "cds.String", "key": True, "notNull": True},
                "REQUIRED_COMPLETIONS": {"type": "cds.Decimal", "precision": 2, "scale": 0}
            }
        },

        "<SCHEMANAME>.VW_CDP_FEATURES": {
            "kind": "entity",
            "@PersonalData.entitySemantics": "DataSubjectDetails",
            "elements": {
                "EMPLOYEE_ID": {"type": "cds.String", "key": True, "notNull": True, "@PersonalData.fieldSemantics": {"#": "DataSubjectIDType"}},
                "CDP_MILESTONE_COMPLETION_RATE": {"type": "cds.Decimal", "precision": 22, "scale": 6},
                "ACTIVE_DEV_GOAL_COUNT": {"type": "cds.Decimal", "precision": 13, "scale": 0}
            }
        },

        "<SCHEMANAME>.VW_LABELED_EMPLOYEES": {
            "kind": "entity",
            "@PersonalData.entitySemantics": "DataSubjectDetails",
            "elements": {
                "EMPLOYEE_ID": {"type": "cds.String", "key": True, "notNull": True, "@PersonalData.fieldSemantics": {"#": "DataSubjectIDType"}},
                "DIGITAL_PROGRAM_COMPLETION_RATE": {"type": "cds.Decimal", "precision": 22, "scale": 6},
                "LEARNING_HOURS_LAST_6MO": {"type": "cds.Decimal", "precision": 38, "scale": 17},
                "LEARNING_VELOCITY_TREND": {"type": "cds.String"},
                "DAYS_SINCE_LAST_LEARNING": {"type": "cds.Decimal", "precision": 9, "scale": 0},
                "CDP_MILESTONE_COMPLETION_RATE": {"type": "cds.Decimal", "precision": 22, "scale": 6},
                "ACTIVE_DEV_GOAL_COUNT": {"type": "cds.Decimal", "precision": 13, "scale": 0},
                "READINESS_LEVEL_DELTA_12MO": {"type": "cds.Double"},
                "HIGH_POTENTIAL_FLAG": {"type": "cds.Decimal", "precision": 1, "scale": 0},
                "LATEST_PERF_RATING": {"type": "cds.Decimal", "precision": 34, "scale": 17},
                "PERF_RATING_DELTA": {"type": "cds.Decimal", "precision": 35, "scale": 17},
                "GOAL_COMPLETION_RATE": {"type": "cds.Decimal", "precision": 22, "scale": 6},
                "SKILL_PROFILE_UPDATES_6MO": {"type": "cds.Decimal", "precision": 13, "scale": 0},
                "READINESS_TIER_LABEL": {"type": "cds.String"}
            }
        },

        "<SCHEMANAME>.VW_LEARNING_FEATURES": {
            "kind": "entity",
            "@PersonalData.entitySemantics": "DataSubjectDetails",
            "elements": {
                "EMPLOYEE_ID": {"type": "cds.String", "key": True, "notNull": True, "@PersonalData.fieldSemantics": {"#": "DataSubjectIDType"}},
                "DIGITAL_PROGRAM_COMPLETION_RATE": {"type": "cds.Decimal", "precision": 22, "scale": 6},
                "LEARNING_HOURS_LAST_6MO": {"type": "cds.Decimal", "precision": 38, "scale": 17},
                "HOURS_RECENT_3MO": {"type": "cds.Decimal", "precision": 38, "scale": 17},
                "HOURS_PRIOR_3MO": {"type": "cds.Decimal", "precision": 38, "scale": 17},
                "DAYS_SINCE_LAST_LEARNING": {"type": "cds.Decimal", "precision": 9, "scale": 0}
            }
        },

        "<SCHEMANAME>.VW_LEARNING_MONTHLY_SERIES": {
            "kind": "entity",
            "elements": {
                "DEPARTMENT": {"type": "cds.Decimal", "precision": 20, "scale": 0, "key": True, "notNull": True},
                "COMPLETIONS": {"type": "cds.Decimal", "precision": 18, "scale": 0},
                "ACTIVE_LEARNERS": {"type": "cds.Decimal", "precision": 18, "scale": 0},
                "TOTAL_CREDIT_HOURS": {"type": "cds.Decimal", "precision": 38, "scale": 17}
            }
        },

        "<SCHEMANAME>.VW_PERFORMANCE_FEATURES": {
            "kind": "entity",
            "@PersonalData.entitySemantics": "DataSubjectDetails",
            "elements": {
                "EMPLOYEE_ID": {"type": "cds.String", "key": True, "notNull": True, "@PersonalData.fieldSemantics": {"#": "DataSubjectIDType"}},
                "LATEST_PERF_RATING": {"type": "cds.Decimal", "precision": 34, "scale": 17},
                "PERF_RATING_DELTA": {"type": "cds.Decimal", "precision": 35, "scale": 17},
                "GOAL_COMPLETION_RATE": {"type": "cds.Decimal", "precision": 22, "scale": 6}
            }
        },

        "<SCHEMANAME>.VW_SKILLS_SIGNAL": {
            "kind": "entity",
            "@PersonalData.entitySemantics": "DataSubjectDetails",
            "elements": {
                "EMPLOYEE_ID": {"type": "cds.String", "key": True, "notNull": True, "@PersonalData.fieldSemantics": {"#": "DataSubjectIDType"}},
                "SKILL_PROFILE_UPDATES_6MO": {"type": "cds.Decimal", "precision": 13, "scale": 0}
            }
        },

        "<SCHEMANAME>.VW_SUCCESSION_FEATURES": {
            "kind": "entity",
            "@PersonalData.entitySemantics": "DataSubjectDetails",
            "elements": {
                "EMPLOYEE_ID": {"type": "cds.String", "key": True, "notNull": True, "@PersonalData.fieldSemantics": {"#": "DataSubjectIDType"}},
                "CURRENT_READINESS_LEVEL": {"type": "cds.String"},
                "READINESS_LEVEL_DELTA_12MO": {"type": "cds.Double"},
                "HIGH_POTENTIAL_FLAG": {"type": "cds.Decimal", "precision": 1, "scale": 0}
            }
        },

        "<SCHEMANAME>.WORKFORCE_READINESS_SCORES": {
            "kind": "entity",
            "@PersonalData.entitySemantics": "DataSubjectDetails",
            "elements": {
                "EMPLOYEE_ID": {"type": "cds.String", "key": True, "notNull": True, "@PersonalData.fieldSemantics": {"#": "DataSubjectIDType"}},
                "DEPARTMENT": {"type": "cds.Decimal", "precision": 20, "scale": 0},
                "JOB_TITLE": {"type": "cds.String"},
                "READINESS_TIER": {"type": "cds.String"},
                "P_READY_NOW": {"type": "cds.Double"},
                "P_FAST_TRACK": {"type": "cds.Double"},
                "P_NEEDS_COACHING": {"type": "cds.Double"},
                "P_NEEDS_PROGRAM": {"type": "cds.Double"},
                "READINESS_VELOCITY_INDEX": {"type": "cds.Double"}
            }
        },

        "<SCHEMANAME>.WORKFORCE_TRANSFORMATION_VELOCITY": {
            "kind": "entity",
            "@PersonalData.entitySemantics": "DataSubjectDetails",
            "elements": {
                "EMPLOYEE_ID": {"type": "cds.String", "key": True, "notNull": True, "@PersonalData.fieldSemantics": {"#": "DataSubjectIDType"}},
                "DEPARTMENT": {"type": "cds.Decimal", "precision": 38, "scale": 0},
                "JOB_TITLE": {"type": "cds.String"},
                "READINESS_TIER": {"type": "cds.String"},
                "READINESS_VELOCITY_INDEX": {"type": "cds.Double"},
                "DEPT_MILESTONE_STATUS": {"type": "cds.String"},
                "DEPT_PROJECTED_COMPLETIONS_12MO": {"type": "cds.Double"}
            }
        }
    }
}

csn_json = json.dumps(csn, separators=(',', ':'))

ord_metadata = json.dumps({
    "title": "<USERNAME> Workforce Transformation Velocity Intelligence",
    "shortDescription": "ML-predicted employee readiness tiers and 12-month department learning trajectory vs transformation milestones.",
    "description": "Combined Snowflake Cortex ML output derived from 8 SAP SuccessFactors data products shared via SAP Business Data Cloud. ML.CLASSIFICATION scores each employee into transformation readiness tiers (READY_NOW, FAST_TRACK_3MO, NEEDS_COACHING, NEEDS_PROGRAM) with a composite velocity index (0-100). ML.FORECAST projects 12-month department-level learning completion trajectories against program milestone gates, classified as ON_TRACK, MARGINAL, POSSIBLE_IF_ACCELERATED, or WILL_MISS."
}, separators=(',', ':'))

print(f'CSN size: {len(csn_json)} bytes, {len(csn["definitions"])-1} entities')
print(f'ORD metadata size: {len(ord_metadata)} bytes')

In [ ]:
%%sql -r publish_result
SELECT SYSTEM$SAP_PUBLISH_DATA_PRODUCT(
    'DB_BDC_SANDBOX.PUBLIC.SAP_BDC_CONNECT_SANDBOX_BFSNOWFLAKE',
    '<SHARENAME>',
    '{{ord_metadata}}',
    '{{csn_json}}'
)

# Done!
Run the notebook to create the data product. Once done, <b>it takes about 10 minutes to publish the data product on the SAP BDC Catalog.</b>